# 02 — U.S. Census counties + browser GeoAI clustering

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/02_census_geoai_counties.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/02_census_geoai_counties.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `d41916702b92605f38b4b255583c4913239496b9488276354bdfc1f22c920719`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 02 — U.S. Census counties + browser GeoAI clustering

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=02_census_geoai_counties.ipynb)**

This notebook uses the Census Bureau's **TIGERweb** GeoJSON service rather than the Census Data API, so it can run without an API key. We create county-level spatial features for Illinois and use scikit-learn K-Means as a lightweight GeoAI pattern.

The model is exploratory: clusters are not policy categories and should not be interpreted as causal or normative labels.

> **JupyterLite compatibility:** this notebook uses `geolibre_lite.LiteMap`, which builds native GeoLibre project/layer definitions and displays them through GeoLibre's hosted `?embed=1` viewer. It deliberately avoids `geolibre.Map`, whose localhost HTTP server cannot bind a socket inside Pyodide/WebAssembly.


## Start here: the question behind the code

**Which counties have similar density and housing profiles?**

**What you will learn:** measure → scale → learn → check. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Browser machine learning: K-Means learns groups from engineered county features.

**Why this is useful:** A small browser model helps explore many counties consistently, then returns group membership to the map for inspection.

**Where the work happens:** Feature engineering, scaling and K-Means run in Pyodide after the Census service returns records.

![From county rows to patterns. Use projected area to turn population counts into comparable densities. Log transforms and standardization reduce domination by feature units and large values. Group similar feature vectors; cluster numbers are category IDs. Compare maps, scatter and correlations before assigning meaning to groups.](assets/learning/02/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: From county rows to patterns — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/02/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/02/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/02/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=02_census_geoai_counties.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/02_census_geoai_counties.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/02/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Measure (0–8 s):** Use projected area to turn population counts into comparable densities. Code landmark: `gdf.to_crs(5070)`.

2. **Scale (8–16 s):** Log transforms and standardization reduce domination by feature units and large values. Code landmark: `StandardScaler()`.

3. **Learn (16–24 s):** Group similar feature vectors; cluster numbers are category IDs. Code landmark: `KMeans(n_clusters=5)`.

4. **Check (24–32 s):** Compare maps, scatter and correlations before assigning meaning to groups. Code landmark: `X.corr()`.


## See the connections

![County patterns: geometry, features, clusters: Geometry: County polygons provide spatial context; Features: Density and ratios answer different questions; Learning: Transform, standardize, then cluster; Check: Cluster IDs have no order or causal meaning](assets/02_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geolibre==3.0.0", "pyodide-http"])
from geolibre_lite import LiteMap as Map


### Code guide 2 — Import table, geometry and learning tools

**Read the Python:** `pd`, `np`, and `gpd` are aliases for pandas, NumPy and GeoPandas. `pyodide_http.patch_all()` adapts requests to browser networking. `StandardScaler` rescales numeric features; `KMeans` learns groups from them.

**Look for:** Tools are ready; no model has been fitted yet.

**Pause and try:** Locate the two learning imports and the one networking adapter.


In [ ]:
import sys, requests, pandas as pd, numpy as np
if sys.platform == "emscripten":
    import pyodide_http
    pyodide_http.patch_all()

import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


### Code guide 3 — Fetch Illinois county records

**Read the Python:** `params` is a dictionary of query options. The STATE filter limits the download, `outFields` selects columns, and `outSR=4326` requests longitude/latitude geometry. `raise_for_status()` stops on HTTP errors; `from_features` creates a spatial table.

**Look for:** `gdf.head()` previews five rows, although `gdf` holds all returned counties.

**Pause and try:** Compare a row's population count with its land area. Why might ranking by count alone favor large counties?


In [ ]:
url = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/57/query"
params = {
    "where": "STATE='17'",
    "outFields": "GEOID,BASENAME,POP100,HU100,AREALAND,AREAWATER",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson",
}
response = requests.get(url, params=params, timeout=60)
response.raise_for_status()
data = response.json()
gdf = gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")
gdf.head()


### Code guide 4 — Learn groups from comparable features

**Read the Python:** `to_crs(5070)` projects geometry before measuring area. Division by one million converts m² to km². `log1p` compresses large values, and `fit_transform` standardizes the features. `fit_predict` fits five groups and returns a group ID per county. The zero fill for an undefined people-per-housing-unit ratio is a modeling choice to revisit if such records occur.

**Look for:** Three density/ratio features plus arbitrary cluster IDs. `random_state=42` makes this fitting reproducible for the same inputs.

**Pause and try:** Try four clusters and rerun the map and charts. Record which counties move without calling any group better or worse.


In [ ]:
# Browser-side feature engineering.
metric = gdf.to_crs(5070)
gdf["area_km2"] = metric.area / 1e6
gdf["population_density"] = gdf["POP100"].astype(float) / gdf["area_km2"]
gdf["housing_density"] = gdf["HU100"].astype(float) / gdf["area_km2"]
gdf["people_per_housing_unit"] = (
    gdf["POP100"].astype(float) / gdf["HU100"].astype(float).replace(0, np.nan)
).fillna(0)

feature_cols = ["population_density", "housing_density", "people_per_housing_unit"]
X = np.log1p(gdf[feature_cols].clip(lower=0))
Xz = StandardScaler().fit_transform(X)
gdf["cluster"] = KMeans(n_clusters=5, random_state=42, n_init=20).fit_predict(Xz)
gdf[["BASENAME", *feature_cols, "cluster"]].sort_values("population_density", ascending=False).head(10)


### Code guide 5 — Compare population and learned patterns

**Read the Python:** A choropleth colors polygons by a column. The first layer uses density; the second uses the learned cluster ID. Quantiles group a numeric distribution into classes. Cluster numbers label categories and do not have a low-to-high meaning.

**Look for:** Two switchable county layers answering different questions.

**Pause and try:** Inspect a county in both layers and describe its actual features rather than only its cluster color.


In [ ]:
m = Map(center=(-89.2, 40.0), zoom=5, height="720px")
m.add_choropleth(
    gdf,
    column="population_density",
    name="2020 population density",
    class_count=7,
    colormap="viridis",
    scheme="quantile",
)
m.add_choropleth(
    gdf,
    column="cluster",
    name="K-Means feature cluster",
    class_count=5,
    colormap="turbo",
    scheme="equal-interval",
)
m


## Why this is GeoAI-relevant

GeoAI is broader than deep neural networks. A common geospatial AI pipeline is:

1. obtain geographic features;
2. engineer spatially meaningful predictors;
3. standardize/transform them;
4. learn a pattern or class;
5. write model outputs back to geometry;
6. inspect the result on an interactive map.

The full `geoai-py` package supports much richer imagery/deep-learning workflows; this browser lab concentrates on the part that can run reliably in WebAssembly.

## Data & software citations

- U.S. Census Bureau TIGERweb State/County MapServer: https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer
- Geometry/attributes are from the Census 2020 county layer exposed by TIGERweb. Source/copyright: U.S. Census Bureau.
- scikit-learn K-Means: https://scikit-learn.org/
- GeoLibre: https://geolibre.app/.

For research use, verify the exact TIGERweb vintage and preserve the query URL/parameters with your output.


### Code guide 6 — Load the plotting tools

**Read the Python:** `from IPython.display import display` lets a cell show objects explicitly. In Pyodide, `await loadPackage('matplotlib')` loads a compiled plotting package. The `if` block is skipped in ordinary Python, where the package must already be installed.

**Look for:** This preparation cell may show no figure; the next cell creates it.

**Pause and try:** Keep this cell before the chart cell when you rerun the notebook after a kernel restart.


In [ ]:
from IPython.display import display
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage("matplotlib")

## Read the clusters in feature space

The map locates each group; the scatter plot shows what the model grouped. Colors below are categorical and cluster numbers are arbitrary. The correlation matrix helps identify redundant features.

### Code guide 7 — Check what the model actually grouped

**Read the Python:** The loop plots one cluster at a time. `symlog` makes a wide numeric range readable while allowing zero. `X.corr()` compares the log-transformed features; the nested loops print each correlation value on the heatmap.

**Look for:** A scatter plot and a correlation matrix that reveal whether population and housing repeat a similar signal.

**Pause and try:** If two predictors are strongly correlated, how might including both affect the clustering?


In [ ]:
from cognition import style_plots, COLORS
import matplotlib.pyplot as plt
style_plots()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for group in sorted(gdf.cluster.unique()):
    part = gdf[gdf.cluster == group]
    axes[0].scatter(part.population_density, part.housing_density,
                    label=f"Cluster {group}", color=COLORS[int(group)], alpha=.8)
axes[0].set(xscale="symlog", yscale="symlog", xlabel="People / km² (symlog)",
            ylabel="Housing units / km² (symlog)", title="Does housing repeat the population signal?")
axes[0].legend(fontsize=8)
corr = X.corr()
im = axes[1].imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
axes[1].set_xticks(range(3), ["Population", "Housing", "People/unit"], rotation=25)
axes[1].set_yticks(range(3), ["Population", "Housing", "People/unit"])
for r in range(3):
    for c in range(3):
        axes[1].text(c, r, f"{corr.iloc[r,c]:.2f}", ha="center", va="center",
                     color="white" if abs(corr.iloc[r,c]) > .65 else "black")
axes[1].set_title("Correlation of log-transformed features")
fig.colorbar(im, ax=axes[1], label="Pearson r")
plt.show()

**Challenge:** rerun K-Means with four clusters. Which counties move, and does that mean their actual population changed? Explain why these associations do not establish causes.

## Make sense of the result

![Is cluster 4 better than cluster 1? The numbers name groups; they do not rank counties. I will describe density and housing instead.](assets/learning/02/cartoon.svg)

**Cartoon transcript:** Is cluster 4 better than cluster 1? → The numbers name groups; they do not rank counties. → I will describe density and housing instead.

**Explain it to a colleague:** Why compare the feature scatter with the geographic map?

<details><summary>Check your reasoning</summary>

The scatter shows what the model grouped; the map shows where those records are. Both are needed to interpret the result.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
